# Exercice 11 : Un mois de taxis à New York

## 1. Chargement et conversion des dates

In [1]:
import ssl
import pandas as pd
import seaborn as sns

ssl._create_default_https_context = ssl._create_unverified_context
taxis = sns.load_dataset("taxis")
taxis["pickup"] = pd.to_datetime(taxis["pickup"])
taxis["dropoff"] = pd.to_datetime(taxis["dropoff"])

print(taxis.shape)
print(taxis["pickup"].min())
print(taxis["pickup"].max())

(6433, 14)
2019-02-28 23:29:03
2019-03-31 23:43:45


6 433 courses, du 28 février 2019 à 23 h 29 au 31 mars 2019. Le 28 février ne compte qu'une course : les données couvrent en pratique le mois de mars.

## 2. Nombre de courses et chiffre d'affaires par jour

In [2]:
courses_par_jour = taxis.set_index("pickup").resample("D")["total"].count()
ca_par_jour = taxis.set_index("pickup").resample("D")["total"].sum()

print(courses_par_jour)
print(ca_par_jour.round(2))

pickup
2019-02-28      1
2019-03-01    241
2019-03-02    198
2019-03-03    169
2019-03-04    171
2019-03-05    228
2019-03-06    257
2019-03-07    218
2019-03-08    235
2019-03-09    204
2019-03-10    185
2019-03-11    209
2019-03-12    218
2019-03-13    244
2019-03-14    260
2019-03-15    201
2019-03-16    220
2019-03-17    178
2019-03-18    173
2019-03-19    201
2019-03-20    233
2019-03-21    221
2019-03-22    229
2019-03-23    209
2019-03-24    149
2019-03-25    155
2019-03-26    178
2019-03-27    232
2019-03-28    205
2019-03-29    209
2019-03-30    215
2019-03-31    187
Freq: D, Name: total, dtype: int64
pickup
2019-02-28       6.30
2019-03-01    4213.83
2019-03-02    3319.02
2019-03-03    3027.32
2019-03-04    3269.08
2019-03-05    4377.38
2019-03-06    4673.61
2019-03-07    4115.70
2019-03-08    4524.21
2019-03-09    3785.80
2019-03-10    3198.69
2019-03-11    4021.26
2019-03-12    3875.80
2019-03-13    4868.06
2019-03-14    4742.55
2019-03-15    4030.29
2019-03-16    3836.58
2

## 3. L'heure de la journée la plus chargée

In [3]:
taxis["heure"] = taxis["pickup"].dt.hour
courses_par_heure = taxis["heure"].value_counts()
print(courses_par_heure.head())

heure
18    417
19    406
17    388
20    368
14    360
Name: count, dtype: int64


18 h est l'heure la plus chargée (417 courses), suivie de 19 h et 17 h : c'est la sortie des bureaux.

## 4. Le jour de la semaine au pourboire moyen le plus élevé

In [4]:
taxis["jour_semaine"] = taxis["pickup"].dt.day_name()
pourboire_par_jour = taxis.groupby("jour_semaine")["tip"].mean().sort_values(ascending=False).round(2)
print(pourboire_par_jour)

jour_semaine
Wednesday    2.09
Thursday     2.06
Monday       2.03
Tuesday      2.00
Friday       1.98
Sunday       1.91
Saturday     1.82
Name: tip, dtype: float64


Le mercredi a le pourboire moyen le plus élevé (2,09 $). Les écarts restent faibles : de 1,82 $ le samedi à 2,09 $ le mercredi. Les jours de semaine donnent un peu plus que le week-end.

## 5. Durée des courses et durées aberrantes

In [5]:
taxis["duree_minutes"] = (taxis["dropoff"] - taxis["pickup"]).dt.total_seconds() / 60
print(taxis["duree_minutes"].describe())

count    6433.000000
mean       14.349617
std        11.643892
min         0.000000
25%         6.500000
50%        10.900000
75%        18.516667
max       107.666667
Name: duree_minutes, dtype: float64


In [6]:
q1 = taxis["duree_minutes"].quantile(0.25)
q3 = taxis["duree_minutes"].quantile(0.75)
ecart = q3 - q1
borne_haute = q3 + 1.5 * ecart

courses_longues = taxis[taxis["duree_minutes"] > borne_haute]
print(f"Borne haute : {borne_haute:.1f} minutes")
print(f"Courses au-dessus : {len(courses_longues)}")
print(courses_longues["distance"].describe())

Borne haute : 36.5 minutes
Courses au-dessus : 358
count    358.000000
mean      13.228575
std        6.121531
min        0.000000
25%        8.787500
50%       12.695000
75%       17.692500
max       36.700000
Name: distance, dtype: float64


In [7]:
courses_nulles = taxis[taxis["duree_minutes"] <= 0]
print(f"Courses de durée nulle ou négative : {len(courses_nulles)}")
print(courses_nulles[["pickup", "dropoff", "distance", "total", "pickup_zone", "dropoff_zone"]])

Courses de durée nulle ou négative : 6
                  pickup             dropoff  distance  total  \
1690 2019-03-22 06:24:14 2019-03-22 06:24:14       0.0   72.0   
5493 2019-03-06 12:14:00 2019-03-06 12:14:00       0.0   10.0   
5624 2019-03-11 14:04:50 2019-03-11 14:04:50       0.0    3.3   
5638 2019-03-13 12:22:00 2019-03-13 12:22:00       0.0   10.0   
6083 2019-03-19 15:34:00 2019-03-19 15:34:00       0.0    5.0   
6356 2019-03-01 16:58:23 2019-03-01 16:58:23       0.0    4.8   

             pickup_zone dropoff_zone  
1690       East New York          NaN  
5493                 NaN          NaN  
5624                 NaN          NaN  
5638                 NaN          NaN  
6083                 NaN          NaN  
6356  Stuyvesant Heights          NaN  


In [8]:
taxis = taxis[taxis["duree_minutes"] > 0]
print(taxis.shape)

(6427, 17)


| Quoi | Pourquoi | Lignes |
|------|----------|--------|
| courses de durée nulle supprimées | départ et arrivée à la même seconde, zone d'arrivée toujours manquante : erreurs d'enregistrement | 6 |
| courses longues conservées | aucune durée négative ni de plusieurs heures (max 108 min) ; ces courses font 13 miles en moyenne, souvent vers ou depuis un aéroport : longues mais réelles | 0 |